In [27]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
import numpy as np
import pandas as pd
from scipy.interpolate import make_interp_spline

#### Implicit FDM scheme

In [28]:
#Defining variables to use later
x_min = -1
x_max = 1
x_steps = 10
t_max = 30 # Hours
t_max = t_max * 3600 # Convert to seconds
t_steps = 10

R = 8.314e-3      # kJ/(mol*K)
T = 1273          # K (1000 °C)

D0_Cu, Q_Cu = 31, 200.3     # mm^2/s, kJ/mol
D0_Ni, Q_Ni = 190, 279.7    # mm^2/s, kJ/mol

D = D0_Cu * np.exp(-Q_Cu / (R * T))   # Cu*  in Cu
# D = D0_Ni * np.exp(-Q_Ni / (R * T)) # Ni*  in Ni
D = 10**-5


In [29]:
def setup(x_min=x_min, x_max=x_max, x_steps=x_steps, t_max=t_max, t_steps=t_steps, D=D):
    x = np.linspace(x_min, x_max, x_steps)
    C_matrix = np.zeros((t_steps, x_steps))
    C_matrix[0, :] = np.heaviside(x, 0.5)   # Using heaviside function to set the first row to 1 for x >= 0 and 0 for x < 0
    dx = (x_max - x_min) / (x_steps - 1)
    dt = t_max / (t_steps - 1)
    r = [D * dt / dx**2]*t_steps
    return C_matrix, dx, dt, r, x

C_matrix, dx, dt, r, x = setup()


In the cell above, a grid of 0s is defined, with the initial condition, C = 1 for positive x and 0 for negative x, is put into the grid. Also, dx, dt and r is defined. The r is defined to make calculations easier, and is a constant for a set time and space step, as well as a constant diffusion coefficiant.

Solving for the next time step for any given space point gives the following:

$$-r\,C_{i-1}^{n+1} + (1+2r)\,C_i^{n+1} - r\,C_{i+1}^{n+1} = C_i^n$$





The cell below solves one step of the implicit system, using a linalg system of this form:


$$
\begin{pmatrix}
1+2r & -2r &  &  &  \\
-r & 1+2r & -r &  & \Large{0} \\
 & \ddots & \ddots & \ddots &  \\
\Large{0} &  & -r & 1+2r & -r \\
 &  &  & -2r & 1+2r
\end{pmatrix}
\begin{pmatrix}
C_0^{n+1} \\ C_1^{n+1} \\ \vdots \\ C_{N-1}^{n+1} \\ C_N^{n+1}
\end{pmatrix}
=
\begin{pmatrix}
C_0^{n} \\ C_1^{n} \\ \vdots \\ C_{N-1}^{n} \\ C_N^{n}
\end{pmatrix}
$$

Here, the 2r points are because of the mirroring of these end points. This is implemented manually in the code.

In [30]:
def implicit_system_one_step(C_matrix, r, step, s = np.zeros(t_steps)):
    C0 = C_matrix[step, :]                      # Takes the step-th row of the C_matrix and stores it in C0
    # print(C0)
    A = np.zeros((len(C0), len(C0)))            # Creates a square matrix of zeros with the same length as C0 to be filled in. This as most of the matrix is 0s anyway.

    for i in range(0, len(C0)):                 # See structure in the markdown cell above. We only fill in the diagonal and the two diagonals next to it. 
        A[i, i] = 2*r[i] + 1
        if i != 0:
            A[i, i-1] = -r[i]+s[i]
        if i != len(C0)-1:
            A[i, i+1] = -r[i] - s[i]
    A[0, 1] = -2*r[0]                              # Adding the end points to fulfill the boundry conditions.
    A[-1, -2] = -2*r[-1]
    np.set_printoptions(precision=3, suppress=True, linewidth=120)
    # print(A)
    C1 = np.linalg.solve(A, C0)                 # Could be replaced by scipy for increase speed.
    # print(C1)
    return C1



In [31]:
def main():
    for step in range(0, t_steps-1):
        C_matrix[step+1, :] = implicit_system_one_step(C_matrix, r, step)
    return C_matrix

C_matrix = main()

In [32]:

plt.imshow(C_matrix, aspect='auto', origin='lower', cmap='gray',
           extent=[x_min, x_max, 0, t_max])
plt.colorbar(label='C')
plt.xlabel('x [mm]')
plt.ylabel('t [s]')
plt.title('Concentration profile C(x,t)')
plt.show()

In [33]:
%matplotlib tk


x = np.linspace(x_min, x_max, x_steps)

fig = plt.figure(figsize=(10, 7))

ax_bar   = fig.add_axes([0.15, 0.80, 0.7, 0.12])   # top: color strip
ax_line  = fig.add_axes([0.15, 0.40, 0.7, 0.32])   # middle: line plot
ax_slider = fig.add_axes([0.15, 0.15, 0.55, 0.04]) # slider
ax_button = fig.add_axes([0.75, 0.14, 0.10, 0.06]) # play/pause button

# top panel: solid concentration strip
img = ax_bar.imshow(C_matrix[[0]], cmap='gray', aspect='auto', vmin=0, vmax=1,
                     extent=[x_min, x_max, 0, 1])
ax_bar.set_yticks([])
ax_bar.set_title('Concentration bar, t = 0 hours')

# middle panel: line plot, updates together with the bar
line, = ax_line.plot(x, C_matrix[0], linewidth=2)
ax_line.set_xlim(x_min, x_max)
ax_line.set_ylim(0, 1)
ax_line.set_xlabel('x [mm]')
ax_line.set_ylabel('C')

slider = Slider(ax_slider, 'timestep', 0, t_steps - 1, valinit=0, valstep=1)
button = Button(ax_button, 'Play')

def update(val):
    n = int(slider.val)
    img.set_data(C_matrix[[n]])
    line.set_ydata(C_matrix[n])
    ax_bar.set_title(f'Concentration bar, t = {n*dt/3600:.2f} hours')
    fig.canvas.draw_idle()

slider.on_changed(update)

playing = [False]

def toggle_play(event):
    playing[0] = not playing[0]
    button.label.set_text('Pause' if playing[0] else 'Play')

button.on_clicked(toggle_play)

total_duration = 15
n_frames = 100          # fewer redraws = more likely to hit the real time budget
interval_ms = int(total_duration * 1000 / n_frames)   # ~150 ms per frame
step_size = max(1, t_steps // n_frames)

def advance():
    if playing[0]:
        n = (int(slider.val) + step_size) % t_steps
        slider.set_val(n)

timer = fig.canvas.new_timer(interval=interval_ms)   # use the variable, not a hardcoded 30
timer.add_callback(advance)
timer.start()

plt.show()

## Part 2 (ii)

Here, we need the r as a function of local composition

$$ \frac{\partial C}{\partial t} = \tilde{D} \frac{\partial ^2 C}{\partial x^2} + \frac{\partial C}{\partial x} \frac{\partial \tilde{D}}{\partial x} $$

The first part corresponds to the same as before. The second part can be implemented in two separate schemes multiplied together. 

$$\frac{C_j^{n+1}-C_j^n}{\Delta t} = \tilde D_j^n\,\frac{C_{j+1}^{n+1}-2C_j^{n+1}+C_{j-1}^{n+1}}{(\Delta x)^2} + \left(\frac{\tilde D_{j+1}^n-\tilde D_{j-1}^n}{2\Delta x}\right)\left(\frac{C_{j+1}^{n+1}-C_{j-1}^{n+1}}{2\Delta x}\right)$$

Solved:
$$-\left(\frac{\tilde D_j^n\,\Delta t}{(\Delta x)^2} - \frac{\Delta t\,(\tilde D_{j+1}^n-\tilde D_{j-1}^n)}{4(\Delta x)^2}\right)C_{j-1}^{n+1} + \left(1+\frac{2\tilde D_j^n\,\Delta t}{(\Delta x)^2}\right)C_j^{n+1} - \left(\frac{\tilde D_j^n\,\Delta t}{(\Delta x)^2} + \frac{\Delta t\,(\tilde D_{j+1}^n-\tilde D_{j-1}^n)}{4(\Delta x)^2}\right)C_{j+1}^{n+1} = C_j^n$$

Defining variables to simplify the expression

$$r_j = \frac{\tilde D_j^n\,\Delta t}{(\Delta x)^2}$$

$$s_j = \frac{\Delta t\,(\tilde D_{j+1}^n-\tilde D_{j-1}^n)}{4(\Delta x)^2}$$

In [34]:
# Importing the dataset from the csv file and sorting them


values = pd.read_csv('D_tilda.csv', sep=';', header=None, decimal=',')
composition = values[0].values
lg_D_tilda  = values[1].values


# Interpolating between the values


lg_D_tilda_spline = make_interp_spline(composition, lg_D_tilda, k=3)

In [35]:
# Taking in one step of concentrations and returning the effective diffusion coefficient D_tilda for each point in space.
def D_tilda(lg_D, C1_vector):
    return D1 * (1 - C1_vector) + D2 * C1_vector   # vektorisert

def D_tilda2(C1_vector):
    return D * (1 - C1_vector) + D * C1_vector   # vektorisert

# Defining the r for each D

def D_tilda_spline_vector(C1_vector):
    lg_D_tilda_vector = lg_D_tilda_spline(C1_vector)
    D_tilda_vector = 10**lg_D_tilda_vector
    return D_tilda_vector

def r_vector(D_tilda_vector, dt, dx):
    return D_tilda_vector * dt / dx**2   # vektorisert, ingen loop nødvendig

# Defining the s for each D

def s_vector(D_tilda_vector, dt, dx):
    s = np.zeros(len(D_tilda_vector))
    s[1:-1] = (D_tilda_vector[2:] - D_tilda_vector[:-2]) * dt / (4 * dx**2)
    return s





With the new quantities defined, the same implicit scheme can be used

In [36]:
def main2():
    for step in range(0, t_steps-1):
        D_tilda_vector = D_tilda_spline_vector(C_matrix[step])
        r = r_vector(D_tilda_vector, dt, dx)
        s = s_vector(D_tilda_vector, dt, dx)
        C_matrix[step+1, :] = implicit_system_one_step(C_matrix, r, step, s)
    return C_matrix

print(main2())



[[0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]]
